In [1]:
import Pkg
Pkg.activate("../chebcoefs")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D/chebcoefs`


In [2]:
using StaticArrays
using FastChebInterp

In [3]:
struct ChebyshevSeries{T, N}
    coefs::Array{T, N}
    lb::SVector{N, T}
    ub::SVector{N, T}
end


function ChebyshevSeries(coefs::Array{T, 1}, lb::T, ub::T) where T
    return ChebyshevSeries(coefs, SVector{1, T}(lb), SVector{1, T}(ub))
end


function ChebyshevSeries(coefs::Array{T, 0}, lb::SVector{0}, ub::SVector{0}) where T
    return coefs[]
end


"""Converts a point `x` to its normalized coordinates in ``[-1, 1]^N``."""
function normalize(f::ChebyshevSeries{T, N}, x::SVector{N, T}) where {T, N}
    @. (2x - f.lb - f.ub) / (f.ub - f.lb)
end


function normalize(f::ChebyshevSeries{T, 1}, x::T) where T
    return normalize(f, SVector{1, T}(x))[]
end


"""Checks if the point `x` is in the domain of `f`."""
function contains(f::ChebyshevSeries{T, N}, x::SVector{N, T}) where {T, N}
    return all(f.lb .≤ x .≤ f.ub)
end


function contains(f::ChebyshevSeries{T, N}, x::AbstractVector{T}) where {T, N}
    return contains(f, SVector{N, T}(x))
end


function contains(f::ChebyshevSeries{T, 1}, x::T) where T
    return contains(f, SVector{1, T}(x))
end


include("clenshaw.jl")
include("gradient.jl")
include("hessian.jl");


######################################################################################################


struct Transformation{F<:Function, G<:Function, H<:Function}
    u::F
    ∇u::G
    Hu::H
end


struct ChebyshevCluster{T, N, M}
    series::NTuple{M, ChebyshevSeries{T, N}}
    tforms::NTuple{M, Transformation}
end


function ChebyshevCluster(series::ChebyshevSeries{T, N}...) where {T, N}
    M = length(series)
    return ChebyshevCluster{T, N, M}(series)
end


function contains(g::ChebyshevCluster{T, N, M}, x::Union{AbstractVector{T}, T}) where {T, N, M}
    for i in 1:M
        u = g.tforms[i].u(x)
        if contains(g.series[i], u)
            return true, i
        end
    end
    return false, nothing
end


function (g::ChebyshevCluster{T, N, M})(x::AbstractVector{T}) where {T, N, M}
    for i in 1:M
        u = g.tforms[i].u(x)
        if contains(g.series[i], u)
            return g.series[i](u)
        end
    end
    throw(DomainError(x))
end


function (g::ChebyshevCluster{T, 1, M})(x::T) where {T, M}
    f = g(SVector{1, T}(x))
    return f
end


function gradient(g::ChebyshevCluster{T, N, M}, x::AbstractVector{T}) where {T, N, M}
    for i in 1:M
        u = g.tforms[i].u(x)
        if contains(g.series[i], u)
            ∇ₓu = g.tforms[i].∇u(x)
            f, ∇ᵤf = gradient(g.series[i], u)
            
            # ∂f/∂x = ∂f/∂u ⋅ ∂u/∂x
            ∇f = ∇ₓu' * ∇ᵤf
            
            return f, ∇f
        end
    end
    throw(DomainError(x))
end


function gradient(g::ChebyshevCluster{T, 1, M}, x::T) where {T, M}
    f, ∇f = gradient(g, SVector{1, T}(x))
    return f, ∇f[]
end;


function hessian(g::ChebyshevCluster{T, N, M}, x::AbstractVector{T}) where {T, N, M}
    for i in 1:M
        u = g.tforms[i].u(x)
        if contains(g.series[i], u)
            ∇ₓu = g.tforms[i].∇u(x)
            Hₓu = g.tforms[i].Hu(x)
            f, ∇ᵤf, Hᵤf = hessian(g.series[i], u)
            
            # ∂f/∂x = ∂f/∂u ⋅ ∂u/∂x
            ∇f = ∇ₓu' * ∇ᵤf
            
            # ∂²f/∂x² = ∂f/∂u ⋅ ∂²u/∂x² + ∂²f/∂u² ⋅ (∂u/∂x)²
            Hf = (reshape(reshape(Hₓu, N, :)' * ∇ᵤf, N, N))' + ∇ₓu' * Hᵤf * ∇ₓu
            
            return f, ∇f, Hf
        end
    end
    throw(DomainError(x))
end


function hessian(g::ChebyshevCluster{T, 1, M}, x::T) where {T, M}
    f, ∇f, Hf = hessian(g, SVector{1, T}(x))
    return f, ∇f[], Hf[]
end;

In [4]:
function f(u)
    x = u.^2
    return sin.(x)
end

function ∇f(u)
    x = u.^2
    return cos.(x)
end

function Hf(u)
    x = u.^2
    return -sin.(x)
end

function u(x::SVector{1, Float64})
    return x.^0.5
end

function ∇u(x::SVector{1, Float64})
    return reshape(0.5*x.^-0.5, 1, 1)
end

function Hu(x::SVector{1, Float64})
    return reshape(-0.25*x.^-1.5, 1, 1, 1)
end

# function u(x::Float64)
#     return x^0.5
# end

# function ∇u(x::Float64)
#     return 0.5*x^-0.5
# end

# function Hu(x::Float64)
#     return -0.25*x^-1.5
# end

x1, x2 = SA[0.0], SA[0.5*π]
u1, u2 = u(x1), u(x2)

n = 50

xc = chebpoints(n, u1[], u2[])
cb = chebinterp(f.(xc), u1[], u2[])

tf = Transformation(u, ∇u, Hu)
cs = ChebyshevSeries(cb.coefs, u1[], u2[])

cc = ChebyshevCluster((cs,), (tf,));

In [5]:
x3 = 0.5*(x1 + x2)
u3 = u(x3);

In [6]:
contains(cc, x3)

(true, 1)

In [7]:
cc(x3[]) - f(u3)[]

1.1102230246251565e-16

In [8]:
g, ∇g = gradient(cc, x3[])

println(g, ", ", f(u3)[])
println(∇g, ", ", ∇f(u3)[])

println(g - f(u3)[])
println(∇g - ∇f(u3)[])

0.7071067811865477, 0.7071067811865476
0.7071067811865487, 0.7071067811865475
1.1102230246251565e-16
1.2212453270876722e-15


In [9]:
g, ∇g, Hg = hessian(cc, x3[])

println(g, ", ", f(u3)[])
println(∇g, ", ", ∇f(u3)[])
println(Hg, ", ", Hf(u3)[])

println(g - f(u3)[])
println(∇g - ∇f(u3)[])
println(Hg - Hf(u3)[])

0.7071067811865477, 0.7071067811865476
0.7071067811865487, 0.7071067811865475
-0.7071067811865537, -0.7071067811865476
1.1102230246251565e-16
1.2212453270876722e-15
-6.106226635438361e-15


In [10]:
function fx(x::SVector{2, Float64})
    return exp(x[1]) * cos(x[2])
end

function f(u::SVector{2, Float64})
    x = u[1]*cos(u[2])
    y = u[1]*sin(u[2])
    return exp(x) * cos(y)
end

function ∇f(u)
    x = u[1]*cos(u[2])
    y = u[1]*sin(u[2])
    return SA[exp(x)*cos(y), -exp(x)*sin(y)]
end

function Hf(u)
    x = u[1]*cos(u[2])
    y = u[1]*sin(u[2])
    return SA[ exp(x)*cos(y) -exp(x)*sin(y);
              -exp(x)*sin(y) -exp(x)*cos(y)]
end

function u(x::SVector{2, Float64})
    return SA[sqrt(x[1]^2 + x[2]^2), atan(x[2]/x[1])]
end

function ∇u(x::SVector{2, Float64})
    r² = x[1]^2 + x[2]^2
    r = √r²
    
    ∇u₁ = SA[ x[1]/r , x[2]/r ]
    ∇u₂ = SA[-x[2]/r², x[1]/r²]
    
    return vcat(∇u₁', ∇u₂')
end

function Hu(x::SVector{2, Float64})
    xy = x[1]*x[2]
    x², y² = x.^2
    r² = x² + y²
    r = √r²
    r³ = r²*r
    r⁴ = r³*r

    Hu₁ = SA[y²/r³  -xy/r³;
             -xy/r³  x²/r³]
    Hu₂ = SA[2xy/r⁴     (y²-x²)/r⁴;
             (y²-x²)/r⁴ -2xy/r⁴   ]

    hess_u = Array{Float64, 3}(undef, 2, 2, 2)
    hess_u[1, :, :] = Hu₁
    hess_u[2, :, :] = Hu₂
    
    return hess_u
end

function u_to_x(u::SVector{2, Float64})
    return SA[u[1]*cos(u[2]), u[1]*sin(u[2])]
end

u1 = SA[0.0, -0.2*π]
u2 = SA[2.0, 0.4*π]

n = (50, 50)

xc = chebpoints(n, u1, u2)
cb = chebinterp(f.(xc), u1, u2)

tf = Transformation(u, ∇u, Hu)
cs = ChebyshevSeries(cb.coefs, u1, u2)

cc = ChebyshevCluster((cs,), (tf,));

In [11]:
# u3 = u1 + 0.3*(u2 - u1)
u3 = SA[0.6, -0.1]
x3 = u_to_x(u3);

In [12]:
Vector{Float64} === Array{Float64, 1}

true

In [13]:
g = cc(x3)

println(g)
println(f(u3))
println()

cc(x3) - f(u3)

1.8134070379886973
1.813407037988697



2.220446049250313e-16

In [14]:
g, ∇g = gradient(cc, x3)

println(g)
println(f(u3))
println()
println(∇g)
println(∇f(u3))
println()

println(g - f(u3))
println(∇g - ∇f(u3))

1.8134070379886973
1.813407037988697

[1.8134070379887013, 0.10875327284161514]
[1.813407037988697, 0.1087532728416087]

2.220446049250313e-16
[4.218847493575595e-15, 6.439293542825908e-15]


In [15]:
g, ∇g, Hg = hessian(cc, x3)

println(g)
println(f(u3))
println()
println(∇g)
println(∇f(u3))
println()
println(Hg)
println(Hf(u3))
println()

println(g - f(u3))
println(∇g - ∇f(u3))
println(Hg - Hf(u3))

1.8134070379886973
1.813407037988697

[1.8134070379887013, 0.10875327284161514]
[1.813407037988697, 0.1087532728416087]

[1.8134070379886607 0.10875327284156461; 0.10875327284156455 -1.8134070379894585]
[1.813407037988697 0.1087532728416087; 0.1087532728416087 -1.813407037988697]

2.220446049250313e-16
[4.218847493575595e-15, 6.439293542825908e-15]
[-3.6415315207705135e-14 -4.408973186542653e-14; -4.414524301665779e-14 -7.613909502879324e-13]
